# Attention U-Net Training: ISLES 2022 + SOOP

Train 3D Attention U-Net with mild augmentation on combined stroke dataset.

**Improvements over baseline U-Net (notebook 03):**
- Attention U-Net architecture (attention gates at skip connections)
- Mild data augmentation (flips, noise, contrast)
- Higher focal loss weight (0.6 vs 0.5) for better small lesion detection

**Previous results:**
- U-Net (ISLES only): batch-avg Dice = 0.606
- U-Net (ISLES+SOOP): batch-avg Dice = 0.705, per-subject Dice = 0.567

**Setup:**
1. Add dataset: `orvile/isles-2022-brain-stoke-dataset`
2. Add input: Your Work -> `03a_download_soop` notebook output (SOOP data)
3. Enable GPU: Settings -> Accelerator -> GPU T4 x2
4. Run all cells

## 1. Setup

In [ ]:
# ===========================================================
# THE ONLY LINE TO CHANGE BETWEEN RUNS: 0, 1, 2, 3 or 4
FOLD = 0

# Guard: refuse to train if SOOP fails to load (see cell below).
EXPECT_SOOP = True
# ===========================================================

# Clone the repo. GitHub is the primary remote -- the GitLab mirror
# lags behind and would train on code without the QC gate.
!git clone https://github.com/Payz111/mri-stroke-assistance.git /kaggle/working/mri-stroke-assist
%cd /kaggle/working/mri-stroke-assist
!git log --oneline -1

In [ ]:
# Install dependencies
!pip install -q monai nibabel SimpleITK pyyaml

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Find datasets

In [ ]:
# Self-contained: a GPU switch restarts the runtime and wipes earlier state,
# so this cell re-imports everything it needs.
import os
import tarfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")


def _has_subjects(path: Path, prefix: str) -> bool:
    try:
        return any(d.name.startswith(prefix) for d in path.iterdir() if d.is_dir())
    except (NotADirectoryError, PermissionError, FileNotFoundError):
        return False


def find_root(prefix: str, reject: str | None = None, max_depth: int = 8) -> Path | None:
    """Breadth-first search for the directory that directly holds <prefix>* dirs.

    Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/, and the
    ISLES archive nests ISLES-2022 inside ISLES-2022, which puts the root six
    levels down -- hence the generous depth.

    Subject folders are never descended into: they cannot contain a root, and
    SOOP has 1323 of them.
    """
    frontier = [INPUT_ROOT]
    for _ in range(max_depth):
        nxt = []
        for node in frontier:
            if _has_subjects(node, prefix) and not (
                reject is not None and _has_subjects(node, reject)
            ):
                return node
            try:
                for child in node.iterdir():
                    if child.is_dir() and not child.name.startswith("sub-"):
                        nxt.append(child)
            except (NotADirectoryError, PermissionError, FileNotFoundError):
                continue
        if not nxt:
            break
        frontier = nxt
    return None


def show_tree(root: Path, max_depth: int = 4, prefix: str = "") -> None:
    """Print the input tree so a wrong path can be diagnosed from the log."""
    if max_depth <= 0:
        return
    try:
        children = sorted(d for d in root.iterdir() if d.is_dir())
    except (NotADirectoryError, PermissionError, FileNotFoundError):
        return
    for child in children[:8]:
        marker = "  (subjects here)" if _has_subjects(child, "sub-") else ""
        print(f"{prefix}  {child.name}/{marker}")
        if not child.name.startswith("sub-"):
            show_tree(child, max_depth - 1, prefix + "  ")


# --- SOOP: either a plain Kaggle Dataset directory or a tar to unpack ---
SOOP_ROOT = None

if Path("/tmp/soop/ds004889").exists() and _has_subjects(Path("/tmp/soop/ds004889"), "sub-"):
    SOOP_ROOT = Path("/tmp/soop/ds004889")
    print(f"SOOP already unpacked at {SOOP_ROOT}")
else:
    # SOOP subjects are sub-1, sub-2...; ISLES are sub-strokecase*. Rejecting the
    # latter keeps the search from settling on ISLES depending on directory order.
    SOOP_ROOT = find_root("sub-", reject="sub-strokecase")

if SOOP_ROOT is None:
    tar_path = next(INPUT_ROOT.rglob("soop_ds004889.tar"), None)
    if tar_path is not None:
        print(f"Unpacking {tar_path} ({tar_path.stat().st_size / 1e9:.1f} GB)...")
        os.makedirs("/tmp/soop", exist_ok=True)
        with tarfile.open(tar_path) as tar:
            tar.extractall("/tmp/soop")
        SOOP_ROOT = Path("/tmp/soop/ds004889")

if SOOP_ROOT is None:
    print("Could not locate the SOOP root (a directory holding sub-* folders).")
    print("Input tree:")
    show_tree(INPUT_ROOT)
    raise FileNotFoundError(
        "SOOP not found. Attach the ds004889 dataset (or the 03a notebook output) as Input."
    )

soop_subs = sorted(d.name for d in SOOP_ROOT.iterdir() if d.name.startswith("sub-"))
print(f"SOOP root    : {SOOP_ROOT}")
print(f"SOOP subjects: {len(soop_subs)}")

# --- ISLES: the archive nests ISLES-2022 one or two levels deep ---
isles_root = find_root("sub-strokecase")
if isles_root is None:
    print("Could not locate a directory containing sub-strokecase* folders.")
    print("Input tree:")
    show_tree(INPUT_ROOT)
    raise FileNotFoundError("ISLES-2022 not found. Attach it in Kaggle settings.")

isles_derivatives = isles_root / "derivatives"
if not isles_derivatives.is_dir():
    found = next((p for p in isles_root.parents if (p / "derivatives").is_dir()), None)
    if found is not None:
        isles_derivatives = found / "derivatives"

isles_subs = sorted(d.name for d in isles_root.iterdir() if d.name.startswith("sub-strokecase"))
print(f"ISLES root       : {isles_root}")
print(f"ISLES derivatives: {isles_derivatives}")
print(f"ISLES subjects   : {len(isles_subs)}")


## 3. Create combined dataset with augmentation

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "/kaggle/working/mri-stroke-assist")

from src.data.isles22_dataset import ISLES22Dataset
from src.data.soop_dataset import SOOPDataset
from src.data.combined_dataset import CombinedStrokeDataset
from src.data.transforms import get_train_transforms, get_val_transforms

# ISLES split for the fold selected in cell 1
split_file = Path(f"/kaggle/working/mri-stroke-assist/data/splits/fold_{FOLD}.json")
with open(split_file) as f:
    split = json.load(f)
print(f"ISLES fold {FOLD}: {split['n_train']} train, {split['n_val']} val")

# Discover SOOP subjects
soop_ds_all = SOOPDataset(data_root=SOOP_ROOT, require_mask=True)
print(f"SOOP subjects with complete data: {len(soop_ds_all)}")

# Training transforms now include augmentation!
train_tfm = get_train_transforms()
val_tfm = get_val_transforms()

# ISLES datasets
isles_train = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="train",
    transform=train_tfm,
)
isles_val = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="val",
    transform=val_tfm,
)

# SOOP training set
soop_train = SOOPDataset(
    data_root=SOOP_ROOT,
    subject_ids=soop_ds_all.subject_ids,
    require_mask=True,
    transform=train_tfm,
)

# Combined
train_combined = CombinedStrokeDataset([isles_train, soop_train])
print(f"\n{train_combined.summary()}")
print(f"Validation (ISLES only): {len(isles_val)}")
print(f"Data increase: {len(isles_train)} -> {len(train_combined)} ({len(train_combined)/len(isles_train):.1f}x)")
print(f"\nAugmentation: RandFlip (L-R 50%, A-P 30%), GaussNoise (20%, std=0.05), Contrast (20%)")

# --- Fail fast rather than train on the wrong data -------------------------
# The first 5-fold attempt silently trained on ISLES alone because SOOP
# resolved to zero subjects and nothing complained. Never again.
if EXPECT_SOOP and len(soop_train) == 0:
    raise RuntimeError(
        "SOOP resolved to 0 subjects, so this run would train on ISLES only and "
        "would not be comparable with the published ISLES+SOOP results. "
        "Set EXPECT_SOOP = False if an ISLES-only run is what you actually want."
    )
print(f"\nSOOP subjects used: {len(soop_train)} (EXPECT_SOOP={EXPECT_SOOP})")


## 4. Train Attention U-Net

In [ ]:
import logging
from torch.utils.data import DataLoader
from src.models.factory import create_model, create_loss
from src.train.trainer import Trainer
from src.train.callbacks import CheckpointCallback, EarlyStoppingCallback

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s", datefmt="%H:%M:%S")

# Training config
EPOCHS = 100
BATCH_SIZE = 4          # Value the published run used -- see Training_results/Train_03_27_2026/experiment_meta.json
LR = 1e-4
PATIENCE = 20
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True          # Mixed precision: ~1.5-2x speedup on T4

print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, patience={PATIENCE}, AMP={USE_AMP}")
print(f"Train: {len(train_combined)}, Val: {len(isles_val)}")
print(f"Device: {DEVICE}")

In [ ]:
# DataLoaders
train_loader = DataLoader(
    train_combined, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    isles_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Attention U-Net model
model_cfg = {
    "name": "attention_unet3d",
    "in_channels": 3,
    "out_channels": 1,
    "features": [32, 64, 128, 256],
    "dropout": 0.1,
}
# Higher focal weight for better small lesion detection
loss_cfg = {
    "type": "dice_focal",
    "dice_weight": 0.4,
    "focal_weight": 0.6,
    "focal_gamma": 2.0,
}

model = create_model(model_cfg)
criterion = create_loss(loss_cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: Attention U-Net 3D")
print(f"Parameters: {n_params:,}")
print(f"Loss: DiceFocal (dice=0.4, focal=0.6, gamma=2.0)")

# Optimizer & scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# Callbacks
output_dir = Path(f"/kaggle/working/outputs/attention_aug_fold{FOLD}")
output_dir.mkdir(parents=True, exist_ok=True)

callbacks = [
    CheckpointCallback(save_dir=output_dir / "checkpoints", monitor="val_dice"),
    EarlyStoppingCallback(patience=PATIENCE, monitor="val_dice"),
]

In [ ]:
import time

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    scheduler=scheduler,
    callbacks=callbacks,
    use_amp=USE_AMP,
)

t0 = time.time()
result = trainer.fit(num_epochs=EPOCHS)
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} min")
print(f"Best val_dice: {result['best_val_dice']:.4f} at epoch {result['best_epoch'] + 1}")
print(f"\n--- Comparison ---")
print(f"U-Net (ISLES only):      batch-avg dice = 0.606")
print(f"U-Net (ISLES+SOOP):      batch-avg dice = 0.705")
print(f"Attn U-Net + aug + AMP:  batch-avg dice = {result['best_val_dice']:.4f}")
print(f"Change vs prev best: {result['best_val_dice'] - 0.705:+.4f}")

## 5. Training curves

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs_range = [h["epoch"] + 1 for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, [h["train_dice"] for h in history], label="Train")
axes[1].plot(epochs_range, [h["val_dice"] for h in history], label="Val")
axes[1].axhline(y=0.606, color="gray", linestyle="--", alpha=0.5, label="ISLES-only (0.606)")
axes[1].axhline(y=0.705, color="r", linestyle="--", alpha=0.5, label="U-Net combined (0.705)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].set_title("Dice Score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(str(output_dir / "training_curves.png"), dpi=150)
plt.show()
print(f"Best val Dice: {result['best_val_dice']:.4f}")

## 6. Save results

In [ ]:
import json

# Save training history
history_path = output_dir / "training_history.json"
with open(history_path, "w") as f:
    json.dump(result["history"], f, indent=2)

# Save experiment metadata
meta = {
    "model": "attention_unet3d",
    "fold": FOLD,
    "features": [32, 64, 128, 256],
    "dropout": 0.1,
    "loss": "dice_focal (0.4/0.6)",
    "augmentation": "RandFlip(LR=0.5, AP=0.3) + GaussNoise(0.05) + Contrast(0.7-1.5)",
    "amp": USE_AMP,
    "epochs_trained": len(history),
    "best_epoch": result["best_epoch"] + 1,
    "best_val_dice": result["best_val_dice"],
    "train_subjects": len(train_combined),
    "val_subjects": len(isles_val),
    "batch_size": BATCH_SIZE,
    "lr": LR,
}
with open(output_dir / "experiment_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Checkpoint: {output_dir / 'checkpoints' / 'best_model.pth'}")
print(f"History: {history_path}")
print(f"Curves: {output_dir / 'training_curves.png'}")
print(f"Metadata: {output_dir / 'experiment_meta.json'}")
print("\nDownload these files from the Output tab!")

# The Output tab does not reliably surface deeply nested files, so drop a copy
# at the top level where it can always be downloaded.
import shutil

flat_ckpt = Path(f"/kaggle/working/best_model_fold{FOLD}.pth")
shutil.copy(output_dir / "checkpoints" / "best_model.pth", flat_ckpt)
shutil.copy(output_dir / "experiment_meta.json", f"/kaggle/working/experiment_meta_fold{FOLD}.json")
shutil.copy(output_dir / "training_history.json", f"/kaggle/working/training_history_fold{FOLD}.json")
print(f"Flat copies for download: {flat_ckpt.name} + meta + history")
